# Score Data Import — UseCase & TargetUser Nodes

Notebook này bổ sung phần còn thiếu trong pipeline hiện tại:
import **score data** (từ quá trình Knowledge Acquisition - Topic 3) vào Neo4j.

## Kiến trúc bổ sung vào KG hiện có:

```
(:Model)-[:SUITABLE_FOR {score: int}]->(:UseCase)
(:Model)-[:TARGETS      {score: int}]->(:TargetUser)
(:Model)-[:HAS_QUALITY]->(q:QualityProfile)
(:Model)-[:HAS_META]->(meta:PhoneMeta)
```

**Yêu cầu:** Đã chạy xong notebook `phone_graphrag_neo4j.ipynb` (Model nodes đã tồn tại)

## 0. Kết nối Neo4j (tái sử dụng config cũ)

In [ ]:
import re
import json
import pandas as pd
from neo4j import GraphDatabase
from typing import Optional

# ── Giữ nguyên config từ notebook gốc ──────────────────────────
NEO4J_URI      = "bolt://172.18.224.1:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "12345678"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("✅ Neo4j connected")

## 1. Định nghĩa Score Fields

Map toàn bộ các trường score từ data đã crawl sang tên node trong KG.
Chia làm 3 nhóm:
- **USE_CASE**: mục đích sử dụng (Gaming, Camera, Văn phòng...)
- **TARGET_USER**: đối tượng người dùng (Sinh viên, Gamer, Doanh nhân...)
- **QUALITY**: chất lượng thực tế camera, pin, phần mềm...

In [ ]:
# ── UseCase fields: key trong data → tên node UseCase ──────────
USE_CASE_FIELDS = {
    'Gaming (Use Case)'             : 'Gaming',
    'Camera / Creator (Use Case)'   : 'Camera & Creator',
    'Văn phòng (Use Case)'          : 'Văn phòng',
    'Mạng xã hội (Use Case)'        : 'Mạng xã hội',
    'Thể thao / Outdoor (Use Case)' : 'Thể thao & Outdoor',
    # Một số record dùng key không có '(Use Case)'
    'Thể thao / Outdoor (Đối tượng)': 'Thể thao & Outdoor',
    'Học tập (Use Case)'            : 'Học tập',
    'Kinh doanh (Use Case)'         : 'Kinh doanh',
}

# ── TargetUser fields: key trong data → tên node TargetUser ────
TARGET_USER_FIELDS = {
    'Học tập (Đối tượng)'       : 'Học tập',
    'Kinh doanh (Đối tượng)'    : 'Kinh doanh',
    'Học sinh cấp 3 (Đối tượng)': 'Học sinh cấp 3',
    'Sinh viên (Đối tượng)'     : 'Sinh viên',
    'Dân văn phòng (Đối tượng)' : 'Dân văn phòng',
    'Freelancer (Đối tượng)'    : 'Freelancer',
    'Gamer (Đối tượng)'         : 'Gamer',
    'Nhiếp ảnh gia (Đối tượng)' : 'Nhiếp ảnh gia',
    'Người lớn tuổi (Đối tượng)': 'Người lớn tuổi',
    'Doanh nhân (Đối tượng)'    : 'Doanh nhân',
}

# ── Meta fields: thông tin bổ sung về sản phẩm ─────────────────
# Lưu trực tiếp vào PhoneMeta node (không phải relationship)
META_FIELDS = [
    'Phân khúc giá',
    'Tuổi đời',
    'Hệ sinh thái',
    'Thương hiệu',
    'Xuất xứ',
    'Giao diện OS',
    'Phiên bản OS ra mắt',
    'Điện thoại gập',
    'Số năm OS update',
    'Điểm giữ giá',
    'Thời gian sạc thực tế (phút)',
    'Pin thực tế (giờ)',
    'AI on-device vs Cloud',
    'AI Score tổng hợp',
    'Chất lượng phần mềm / ít bloatware',
]

# ── Quality fields: điểm chất lượng camera/video ───────────────
QUALITY_FIELDS = {
    'Chất lượng ảnh ban ngày'   : 'photo_day_score',
    'Chất lượng ảnh ban đêm'    : 'photo_night_score',
    'Chất lượng video thực tế'  : 'video_score',
    'Chất lượng selfie thực tế' : 'selfie_score',
}

# ── Spec fields trực tiếp trên Model node ──────────────────────
# Các trường này sẽ được SET thẳng vào Model node để query nhanh
MODEL_DIRECT_FIELDS = {
    'Màu sắc hỗ trợ'                  : 'color_count',
    'Số lượng camera sau (thật)'       : 'rear_camera_count',
    'Độ phân giải camera chính'        : 'main_camera_resolution',
    'Aperture camera chính'            : 'main_camera_aperture',
    'OIS / Chống rung'                 : 'ois',
    'Zoom quang học'                   : 'optical_zoom',
    'Quay video camera sau'            : 'video_resolution',
    'Độ phân giải camera trước'        : 'front_camera_resolution',
    'Kích thước màn hình'              : 'screen_size',
    'Công nghệ màn hình'               : 'screen_tech',
    'Tần số quét'                      : 'refresh_rate',
    'Độ sáng tối đa'                   : 'brightness_nit',
    'Kiểu màn hình / Notch'            : 'notch_type',
    'Chip tier'                        : 'chip_name',
    'Xếp loại chip'                    : 'chip_grade',
    'AnTuTu score'                     : 'antutu_score',
    'RAM'                              : 'ram',
    'Bộ nhớ trong'                     : 'storage',
    'Dung lượng pin'                   : 'battery_mah',
    'Sạc có dây'                       : 'charging_wired_w',
    'Sạc không dây'                    : 'charging_wireless_w',
    'Mạng di động'                     : 'network',
    'Wi-Fi'                            : 'wifi',
    'Bluetooth'                        : 'bluetooth',
    'NFC'                              : 'nfc',
    'Jack 3.5mm'                       : 'jack_35mm',
    'Hỗ trợ SIM'                       : 'sim_support',
    'eSIM'                             : 'esim',
    'Kháng nước / bụi'                 : 'water_resistance',
    'Chất lượng mặt lưng'              : 'back_material',
    'Chất lượng khung viền'            : 'frame_material',
    'Độ mỏng'                          : 'thickness_mm',
    'Trọng lượng'                      : 'weight_g',
    'Cảm biến vân tay'                 : 'fingerprint_sensor',
    'Nhận diện khuôn mặt'              : 'face_recognition',
}

print(f"✅ Defined:")
print(f"   {len(USE_CASE_FIELDS)} UseCase fields")
print(f"   {len(TARGET_USER_FIELDS)} TargetUser fields")
print(f"   {len(META_FIELDS)} Meta fields")
print(f"   {len(QUALITY_FIELDS)} Quality fields")
print(f"   {len(MODEL_DIRECT_FIELDS)} direct Model property fields")

## 2. Helper: Parse Score Value

Score có thể là string `'4'`, `'4 Rất tốt cho phân khúc'`, int `4`, hoặc `'N/A'`.
Hàm này chuẩn hóa tất cả về int 1-5 hoặc None.

In [ ]:
def parse_score(value) -> Optional[int]:
    """
    Chuẩn hóa score về int 1-5.
    Xử lý các dạng:
    - '4'              → 4
    - '4 Rất tốt...'  → 4  (extract số đầu tiên)
    - 4                → 4
    - 'N/A' / None     → None
    - 'Không đề cập'  → None
    """
    if value is None:
        return None
    s = str(value).strip()
    if s in ('N/A', '', 'Không đề cập', 'nan'):
        return None
    # Extract số đầu tiên
    m = re.match(r'^(\d+)', s)
    if m:
        score = int(m.group(1))
        return score if 1 <= score <= 5 else None
    return None


def normalize_na(value) -> Optional[str]:
    """Chuẩn hóa N/A về None, các giá trị khác giữ nguyên dạng string."""
    if value is None:
        return None
    s = str(value).strip()
    if s in ('N/A', '', 'nan', 'None'):
        return None
    return s


def parse_model_id(sku: str) -> str:
    """Bóc model_id từ sku (bỏ storage suffix như -256gb, -512gb, -1tb)."""
    return re.sub(r'-\d+(gb|tb)$', '', str(sku).strip().lower(), flags=re.IGNORECASE)


# Test
assert parse_score('4') == 4
assert parse_score('4 Rất tốt cho phân khúc') == 4
assert parse_score('N/A') is None
assert parse_score('Không đề cập') is None
assert parse_model_id('iphone-17-pro-max-512gb') == 'iphone-17-pro-max'
assert parse_model_id('iphone-17-pro-max') == 'iphone-17-pro-max'
print("✅ Helpers ready")

## 3. Tạo Constraints & Indexes

In [ ]:
CONSTRAINTS = [
    # UseCase node: unique theo name
    "CREATE CONSTRAINT use_case_name IF NOT EXISTS "
    "FOR (u:UseCase) REQUIRE u.name IS UNIQUE",

    # TargetUser node: unique theo name
    "CREATE CONSTRAINT target_user_name IF NOT EXISTS "
    "FOR (t:TargetUser) REQUIRE t.name IS UNIQUE",

    # QualityProfile: unique theo model_id
    "CREATE CONSTRAINT quality_profile_id IF NOT EXISTS "
    "FOR (q:QualityProfile) REQUIRE q.model_id IS UNIQUE",

    # PhoneMeta: unique theo model_id
    "CREATE CONSTRAINT phone_meta_id IF NOT EXISTS "
    "FOR (m:PhoneMeta) REQUIRE m.model_id IS UNIQUE",
]

INDEXES = [
    # Index để query nhanh theo score
    "CREATE INDEX use_case_score IF NOT EXISTS "
    "FOR ()-[r:SUITABLE_FOR]-() ON (r.score)",

    "CREATE INDEX target_user_score IF NOT EXISTS "
    "FOR ()-[r:TARGETS]-() ON (r.score)",

    # Index trên Model để filter nhanh
    "CREATE INDEX model_price_segment IF NOT EXISTS "
    "FOR (m:Model) ON (m.price_segment)",

    "CREATE INDEX model_ecosystem IF NOT EXISTS "
    "FOR (m:Model) ON (m.ecosystem)",

    "CREATE INDEX model_weight IF NOT EXISTS "
    "FOR (m:Model) ON (m.weight_g)",
]

with driver.session() as session:
    for stmt in CONSTRAINTS:
        session.run(stmt)
        print(f"  ✓ {stmt[:60]}...")
    for stmt in INDEXES:
        try:
            session.run(stmt)
            print(f"  ✓ {stmt[:60]}...")
        except Exception as e:
            # Relationship indexes cần Neo4j Enterprise hoặc 5.x+
            print(f"  ⚠️  Skip index (may need Neo4j 5.x): {e}")

print("\n✅ Constraints & indexes ready")

## 4. Upsert Functions

In [ ]:
# ── 4.1 UseCase node ─────────────────────────────────────────────
def upsert_use_case(tx, name: str):
    """
    Tạo UseCase node nếu chưa tồn tại.
    UseCase là shared node — nhiều Model cùng trỏ đến.
    Ví dụ: UseCase {name: 'Gaming'} dùng chung cho mọi model.
    """
    tx.run("""
        MERGE (u:UseCase {name: $name})
    """, {'name': name})


# ── 4.2 TargetUser node ──────────────────────────────────────────
def upsert_target_user(tx, name: str):
    """
    Tạo TargetUser node nếu chưa tồn tại.
    TargetUser là shared node — dùng chung cho mọi model.
    Ví dụ: TargetUser {name: 'Gamer'}
    """
    tx.run("""
        MERGE (t:TargetUser {name: $name})
    """, {'name': name})


# ── 4.3 SUITABLE_FOR relationship (Model → UseCase) ───────────────
def upsert_suitable_for(tx, model_id: str, use_case_name: str, score: int):
    """
    Tạo/update relationship SUITABLE_FOR với score.
    Score 1-5 do LLM sinh ra từ specs + description (Topic 3 KA).
    """
    tx.run("""
        MATCH (m:Model {model_id: $model_id})
        MATCH (u:UseCase {name: $use_case_name})
        MERGE (m)-[r:SUITABLE_FOR]->(u)
        SET r.score = $score
    """, {
        'model_id'      : model_id,
        'use_case_name' : use_case_name,
        'score'         : score,
    })


# ── 4.4 TARGETS relationship (Model → TargetUser) ─────────────────
def upsert_targets(tx, model_id: str, target_name: str, score: int):
    """
    Tạo/update relationship TARGETS với score.
    """
    tx.run("""
        MATCH (m:Model {model_id: $model_id})
        MATCH (t:TargetUser {name: $target_name})
        MERGE (m)-[r:TARGETS]->(t)
        SET r.score = $score
    """, {
        'model_id'   : model_id,
        'target_name': target_name,
        'score'      : score,
    })


# ── 4.5 QualityProfile node (gắn vào Model) ───────────────────────
def upsert_quality_profile(tx, model_id: str, quality: dict):
    """
    QualityProfile lưu điểm chất lượng thực tế:
    photo_day, photo_night, video, selfie (1-5).
    Tách thành node riêng để query theo chất lượng camera độc lập.
    """
    tx.run("""
        MATCH (m:Model {model_id: $model_id})
        MERGE (q:QualityProfile {model_id: $model_id})
        SET q.photo_day_score   = $photo_day,
            q.photo_night_score = $photo_night,
            q.video_score       = $video,
            q.selfie_score      = $selfie,
            q.avg_camera_score  = $avg_camera
        MERGE (m)-[:HAS_QUALITY]->(q)
    """, {
        'model_id'   : model_id,
        'photo_day'  : quality.get('photo_day_score'),
        'photo_night': quality.get('photo_night_score'),
        'video'      : quality.get('video_score'),
        'selfie'     : quality.get('selfie_score'),
        'avg_camera' : quality.get('avg_camera_score'),
    })


# ── 4.6 PhoneMeta node (gắn vào Model) ───────────────────────────
def upsert_phone_meta(tx, model_id: str, meta: dict):
    """
    PhoneMeta lưu thông tin context:
    phân khúc, hệ sinh thái, OS update, điểm giữ giá, AI score...
    Tách thành node riêng để dễ filter theo ecosystem, segment.
    """
    tx.run("""
        MATCH (m:Model {model_id: $model_id})
        MERGE (meta:PhoneMeta {model_id: $model_id})
        SET meta.price_segment       = $price_segment,
            meta.ecosystem           = $ecosystem,
            meta.brand               = $brand,
            meta.origin              = $origin,
            meta.os_version          = $os_version,
            meta.os_update_years     = $os_update_years,
            meta.is_foldable         = $is_foldable,
            meta.price_retention     = $price_retention,
            meta.charging_time_min   = $charging_time_min,
            meta.battery_life_hours  = $battery_life_hours,
            meta.ai_processing       = $ai_processing,
            meta.ai_score            = $ai_score,
            meta.software_quality    = $software_quality
        MERGE (m)-[:HAS_META]->(meta)
    """, {
        'model_id'         : model_id,
        'price_segment'    : meta.get('price_segment'),
        'ecosystem'        : meta.get('ecosystem'),
        'brand'            : meta.get('brand'),
        'origin'           : meta.get('origin'),
        'os_version'       : meta.get('os_version'),
        'os_update_years'  : meta.get('os_update_years'),
        'is_foldable'      : meta.get('is_foldable'),
        'price_retention'  : meta.get('price_retention'),
        'charging_time_min': meta.get('charging_time_min'),
        'battery_life_hours': meta.get('battery_life_hours'),
        'ai_processing'    : meta.get('ai_processing'),
        'ai_score'         : meta.get('ai_score'),
        'software_quality' : meta.get('software_quality'),
    })


# ── 4.7 Cập nhật direct properties trên Model node ───────────────
def update_model_direct_props(tx, model_id: str, props: dict):
    """
    Set thêm các property thường dùng để filter trực tiếp trên Model node,
    tránh phải traverse sang node khác khi query đơn giản.
    Ví dụ: weight_g, nfc, network, ecosystem, price_segment.
    """
    if not props:
        return
    # Build dynamic SET clause
    set_clauses = ', '.join([f'm.{k} = ${k}' for k in props.keys()])
    params = {'model_id': model_id, **props}
    tx.run(f"""
        MATCH (m:Model {{model_id: $model_id}})
        SET {set_clauses}
    """, params)


print("✅ Upsert functions ready")

## 5. Khởi tạo UseCase & TargetUser Nodes (shared nodes)

Tạo sẵn tất cả UseCase và TargetUser nodes.
Đây là **shared nodes** — tồn tại một lần, nhiều Model cùng kết nối đến.

In [ ]:
with driver.session() as session:
    # Tạo UseCase nodes
    print("Creating UseCase nodes...")
    unique_use_cases = set(USE_CASE_FIELDS.values())
    for uc_name in sorted(unique_use_cases):
        session.execute_write(upsert_use_case, uc_name)
        print(f"  ✓ UseCase: {uc_name}")

    # Tạo TargetUser nodes
    print("\nCreating TargetUser nodes...")
    unique_targets = set(TARGET_USER_FIELDS.values())
    for tu_name in sorted(unique_targets):
        session.execute_write(upsert_target_user, tu_name)
        print(f"  ✓ TargetUser: {tu_name}")

print("\n✅ All shared nodes created")

## 6. Load Score Data

Score data được lưu dưới dạng list of dicts (giống `data_apple_mobile` hay `masstel_data`).
Mỗi dict có key `sku` và các trường score đã định nghĩa ở trên.

> **Thay thế `ALL_SCORE_DATA` bên dưới bằng data thực của bạn.**

In [ ]:
# ── THAY THẾ bằng data thực của bạn ──────────────────────────────
# Có thể merge nhiều list:
#   ALL_SCORE_DATA = data_apple_mobile + masstel_data + samsung_data + ...
#
# Hoặc load từ file:
#   import json
#   with open('score_data.json') as f:
#       ALL_SCORE_DATA = json.load(f)

# Ví dụ với 1 record để test:
ALL_SCORE_DATA = [
    {
        'name': 'iPhone 17 Pro Max 256GB',
        'sku' : 'iphone-17-pro-max',
        'Phân khúc giá'                  : 'Cao cấp',
        'Tuổi đời'                       : '8 tháng',
        'Hệ sinh thái'                   : 'iOS',
        'Thương hiệu'                    : 'Apple',
        'Xuất xứ'                        : 'Mỹ',
        'Giao diện OS'                   : 'iOS',
        'Phiên bản OS ra mắt'            : 'iOS 26',
        'Điện thoại gập'                 : 'không gập',
        'Số năm OS update'               : 'Tối thiểu 6 năm',
        'Điểm giữ giá'                   : 'Cao (~70-85%)',
        'Thời gian sạc thực tế (phút)'   : 'Không đề cập',
        'Pin thực tế (giờ)'              : 'Không đề cập',
        'AI on-device vs Cloud'          : 'On-device (Apple A-series)',
        'AI Score tổng hợp'              : '5',
        'Chất lượng phần mềm / ít bloatware': 'Rất tốt - không bloatware',
        # UseCase scores
        'Gaming (Use Case)'              : '3',
        'Camera / Creator (Use Case)'    : '4',
        'Văn phòng (Use Case)'           : '3',
        'Mạng xã hội (Use Case)'         : '4',
        'Thể thao / Outdoor (Đối tượng)' : '5',
        # TargetUser scores
        'Học tập (Đối tượng)'            : '4',
        'Kinh doanh (Đối tượng)'         : '5',
        'Học sinh cấp 3 (Đối tượng)'     : '4',
        'Sinh viên (Đối tượng)'          : '4',
        'Dân văn phòng (Đối tượng)'      : '4',
        'Freelancer (Đối tượng)'         : '4',
        'Gamer (Đối tượng)'              : '5',
        'Nhiếp ảnh gia (Đối tượng)'      : '4',
        'Người lớn tuổi (Đối tượng)'     : '3',
        'Doanh nhân (Đối tượng)'         : '5',
        # Quality scores
        'Chất lượng ảnh ban ngày'        : '5',
        'Chất lượng ảnh ban đêm'         : '5',
        'Chất lượng video thực tế'       : '5',
        'Chất lượng selfie thực tế'      : '5',
        # Spec fields
        'Màu sắc hỗ trợ'                 : 3,
        'Số lượng camera sau (thật)'     : 3,
        'Độ phân giải camera chính'      : '48MP',
        'Kháng nước / bụi'               : 'IP68',
        'Trọng lượng'                    : '231 g',
        'Độ mỏng'                        : '8.75 mm',
        'NFC'                            : 'Có',
        'Mạng di động'                   : '5G',
        'Cảm biến vân tay'               : 'N/A',
        'Nhận diện khuôn mặt'            : '3D Face ID',
    },
    # Thêm data của bạn ở đây...
]

print(f"✅ Loaded {len(ALL_SCORE_DATA)} records")
print(f"   Sample keys: {list(ALL_SCORE_DATA[0].keys())[:5]}...")

## 7. Main Import Loop — Score Data

In [ ]:
errors   = []
imported = []
skipped  = []  # Model chưa tồn tại trong KG

total = len(ALL_SCORE_DATA)

with driver.session() as session:

    for idx, item in enumerate(ALL_SCORE_DATA):
        sku      = item.get('sku', '')
        name     = item.get('name', sku)
        model_id = parse_model_id(sku)

        print(f"\n[{idx+1}/{total}] {model_id}")

        try:
            # ── Kiểm tra Model đã tồn tại chưa ──────────────────
            check = session.run(
                "MATCH (m:Model {model_id: $id}) RETURN m.name AS name",
                {'id': model_id}
            ).single()

            if not check:
                print(f"  ⚠️  Model not found in KG — skipping")
                skipped.append(sku)
                continue

            # ── 1. UseCase scores → SUITABLE_FOR relationships ──
            uc_count = 0
            for field_key, uc_name in USE_CASE_FIELDS.items():
                score = parse_score(item.get(field_key))
                if score is not None:
                    session.execute_write(upsert_suitable_for, model_id, uc_name, score)
                    uc_count += 1
            print(f"  ✓ {uc_count} UseCase scores")

            # ── 2. TargetUser scores → TARGETS relationships ─────
            tu_count = 0
            for field_key, tu_name in TARGET_USER_FIELDS.items():
                score = parse_score(item.get(field_key))
                if score is not None:
                    session.execute_write(upsert_targets, model_id, tu_name, score)
                    tu_count += 1
            print(f"  ✓ {tu_count} TargetUser scores")

            # ── 3. Quality scores → QualityProfile node ─────────
            quality = {}
            scores_list = []
            for field_key, prop_name in QUALITY_FIELDS.items():
                score = parse_score(item.get(field_key))
                quality[prop_name] = score
                if score is not None:
                    scores_list.append(score)
            quality['avg_camera_score'] = (
                round(sum(scores_list) / len(scores_list), 2)
                if scores_list else None
            )
            session.execute_write(upsert_quality_profile, model_id, quality)
            print(f"  ✓ QualityProfile (avg={quality['avg_camera_score']})")

            # ── 4. Meta → PhoneMeta node ─────────────────────────
            meta = {
                'price_segment'    : normalize_na(item.get('Phân khúc giá')),
                'ecosystem'        : normalize_na(item.get('Hệ sinh thái')),
                'brand'            : normalize_na(item.get('Thương hiệu')),
                'origin'           : normalize_na(item.get('Xuất xứ')),
                'os_version'       : normalize_na(item.get('Phiên bản OS ra mắt')),
                'os_update_years'  : normalize_na(item.get('Số năm OS update')),
                'is_foldable'      : item.get('Điện thoại gập', '') != 'không gập',
                'price_retention'  : normalize_na(item.get('Điểm giữ giá')),
                'charging_time_min': normalize_na(item.get('Thời gian sạc thực tế (phút)')),
                'battery_life_hours': normalize_na(item.get('Pin thực tế (giờ)')),
                'ai_processing'    : normalize_na(item.get('AI on-device vs Cloud')),
                'ai_score'         : parse_score(item.get('AI Score tổng hợp')),
                'software_quality' : normalize_na(item.get('Chất lượng phần mềm / ít bloatware')),
            }
            session.execute_write(upsert_phone_meta, model_id, meta)
            print(f"  ✓ PhoneMeta ({meta['price_segment']}, {meta['ecosystem']})")

            # ── 5. Direct properties trên Model node ─────────────
            direct_props = {}
            for field_key, prop_name in MODEL_DIRECT_FIELDS.items():
                val = normalize_na(item.get(field_key))
                if val is not None:
                    direct_props[prop_name] = val

            # Thêm shortcut properties quan trọng để filter nhanh
            if meta.get('price_segment'):
                direct_props['price_segment'] = meta['price_segment']
            if meta.get('ecosystem'):
                direct_props['ecosystem'] = meta['ecosystem']

            if direct_props:
                session.execute_write(update_model_direct_props, model_id, direct_props)
                print(f"  ✓ {len(direct_props)} direct properties updated")

            imported.append(sku)

        except Exception as e:
            import traceback
            print(f"  ❌ ERROR: {e}")
            traceback.print_exc()
            errors.append({'sku': sku, 'error': str(e)})


print(f"\n{'='*50}")
print(f"✅ Done: {len(imported)}/{total} records imported")
if skipped:
    print(f"⚠️  {len(skipped)} skipped (Model not in KG): {skipped}")
if errors:
    print(f"❌ {len(errors)} errors:")
    for e in errors:
        print(f"   {e['sku']}: {e['error']}")

## 8. Verification Queries

Kiểm tra data đã import đúng chưa.

In [ ]:
with driver.session() as session:

    print("=" * 60)
    print("Query 1: Tất cả UseCase nodes")
    print("=" * 60)
    result = session.run("MATCH (u:UseCase) RETURN u.name ORDER BY u.name")
    for r in result:
        print(f"  • {r['u.name']}")

    print()
    print("=" * 60)
    print("Query 2: Tất cả TargetUser nodes")
    print("=" * 60)
    result = session.run("MATCH (t:TargetUser) RETURN t.name ORDER BY t.name")
    for r in result:
        print(f"  • {r['t.name']}")

    print()
    print("=" * 60)
    print("Query 3: Điện thoại phù hợp Gaming (score >= 4)")
    print("=" * 60)
    result = session.run("""
        MATCH (m:Model)-[r:SUITABLE_FOR]->(u:UseCase {name: 'Gaming'})
        WHERE r.score >= 4
        RETURN m.name AS model, r.score AS score
        ORDER BY r.score DESC
    """)
    for r in result:
        print(f"  [{r['score']}/5] {r['model']}")

    print()
    print("=" * 60)
    print("Query 4: Điện thoại tốt nhất cho Sinh viên")
    print("=" * 60)
    result = session.run("""
        MATCH (m:Model)-[r:TARGETS]->(t:TargetUser {name: 'Sinh viên'})
        RETURN m.name AS model, r.score AS score
        ORDER BY r.score DESC
        LIMIT 5
    """)
    for r in result:
        print(f"  [{r['score']}/5] {r['model']}")

    print()
    print("=" * 60)
    print("Query 5: Điện thoại nhỏ gọn + phù hợp Văn phòng")
    print("(use case thực tế: chatbot sẽ generate query này)")
    print("=" * 60)
    result = session.run("""
        MATCH (m:Model)-[r:SUITABLE_FOR]->(u:UseCase {name: 'Văn phòng'})
        WHERE r.score >= 3
        MATCH (m)-[:HAS_META]->(meta:PhoneMeta)
        RETURN m.name       AS model,
               r.score      AS office_score,
               m.weight_g   AS weight,
               m.thickness_mm AS thickness,
               meta.price_segment AS segment
        ORDER BY r.score DESC, m.weight_g ASC
        LIMIT 5
    """)
    for r in result:
        print(f"  [{r['office_score']}/5] {r['model']} | {r['weight']}g | {r['thickness']} | {r['segment']}")

    print()
    print("=" * 60)
    print("Query 6: Camera quality ranking")
    print("=" * 60)
    result = session.run("""
        MATCH (m:Model)-[:HAS_QUALITY]->(q:QualityProfile)
        RETURN m.name              AS model,
               q.photo_day_score  AS day,
               q.photo_night_score AS night,
               q.video_score      AS video,
               q.selfie_score     AS selfie,
               q.avg_camera_score AS avg
        ORDER BY q.avg_camera_score DESC
        LIMIT 5
    """)
    for r in result:
        print(f"  avg={r['avg']} | {r['model']} "
              f"(day={r['day']}, night={r['night']}, video={r['video']}, selfie={r['selfie']})")

## 9. Tóm tắt Schema sau khi import

Sau khi chạy xong cả 2 notebooks, KG có cấu trúc đầy đủ:

```
(:Brand)
  └─[:HAS_SERIES]─> (:Series)
       └─[:HAS_MODEL]─> (:Model)  ← node trung tâm
            ├─[:HAS_CONFIG]──>  (:ModelStorage)   # giá theo dung lượng
            ├─[:HAS_VARIANT]──> (:Variant)
            │    ├─[:OF_CONFIG]──> (:ModelStorage)
            │    └─[:HAS_COLOR]──> (:ColorVariant)
            ├─[:HAS_SPEC]────>  (:SpecCategory)
            │    └─[:HAS_SPEC_ITEM]──> (:SpecItem)  # ống kính camera
            │
            │  ← BỔ SUNG TỪ NOTEBOOK NÀY ↓
            ├─[:SUITABLE_FOR {score}]──> (:UseCase)      # UseCase scores
            ├─[:TARGETS {score}]──────> (:TargetUser)    # TargetUser scores
            ├─[:HAS_QUALITY]──────────> (:QualityProfile) # camera/video scores
            ├─[:HAS_META]─────────────> (:PhoneMeta)    # ecosystem, segment...
            └─[:UPGRADE_OF]───────────> (:Model)         # liên kết đời trước
```

### Cypher query mẫu cho chatbot:

```cypher
// "Tìm máy gaming dưới 10 triệu"
MATCH (m:Model)-[r:SUITABLE_FOR]->(u:UseCase {name: 'Gaming'})
MATCH (m)-[:HAS_CONFIG]->(ms:ModelStorage)
WHERE r.score >= 4 AND ms.sale_price <= 10000000
RETURN m.name, r.score, ms.sale_price
ORDER BY r.score DESC, ms.sale_price ASC

// "Máy nhỏ gọn cho văn phòng, iOS"
MATCH (m:Model)-[r:SUITABLE_FOR]->(u:UseCase {name: 'Văn phòng'})
WHERE m.ecosystem = 'iOS' AND m.weight_g < 185
RETURN m.name, r.score, m.weight_g
ORDER BY r.score DESC
```

## 10. Close Connection

In [ ]:
driver.close()
print("✅ Neo4j connection closed")
print()
print("📊 Summary:")
print(f"   Records processed : {total}")
print(f"   Successfully imported: {len(imported)}")
print(f"   Skipped (no Model) : {len(skipped)}")
print(f"   Errors             : {len(errors)}")